# Module 4 - Class 5: Regularization Study
**Khamidullokhon Abduvokhidov**

In [ ]:
# Prepare scaled, one-hot-encoded Telco data for logistic regression.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
df = pd.read_csv('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
y = df['Churn'].map({'No': 0, 'Yes': 1})
X = pd.get_dummies(df.drop(columns=['customerID', 'Churn']), drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler(); X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index); X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [ ]:
# Fit unregularized, L1, and L2 logistic regression models.
model_a = LogisticRegression(penalty=None, max_iter=1000).fit(X_train, y_train)
model_b = LogisticRegression(penalty='l1', solver='saga', C=1.0, max_iter=1000).fit(X_train, y_train)
model_c = LogisticRegression(penalty='l2', C=1.0, max_iter=1000).fit(X_train, y_train)
def evaluate(model, name):
    pred, proba = model.predict(X_test), model.predict_proba(X_test)[:, 1]
    return {'Model':name,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred),'Recall':recall_score(y_test,pred),'F1':f1_score(y_test,pred),'AUC':roc_auc_score(y_test,proba)}
results = pd.DataFrame([evaluate(model_a,'No Regularization'), evaluate(model_b,'L1 (Lasso)'), evaluate(model_c,'L2 (Ridge)')]).set_index('Model')
display(results)

In [ ]:
# Compare coefficient shrinkage and identify L1 feature selection.
coef_df = pd.DataFrame({'Feature':X_train.columns,'No Reg':model_a.coef_[0],'L1':model_b.coef_[0],'L2':model_c.coef_[0]})
plt.figure(figsize=(10,12)); sns.heatmap(coef_df.set_index('Feature'), cmap='coolwarm', center=0); plt.title('Coefficient Comparison'); plt.show()
for name, model in [('No Reg',model_a),('L1',model_b),('L2',model_c)]: print(f'{name}: {np.sum(model.coef_[0] != 0)} non-zero coefficients')
print('L1 zeroed features:', coef_df.loc[coef_df['L1'] == 0, 'Feature'].tolist())

In [ ]:
# Vary L2 strength and plot the performance-complexity tradeoff.
C_values = [0.001, 0.01, 0.1, 1, 10, 100]; f1_scores = []; n_large_coefs = []
for C in C_values:
    model = LogisticRegression(penalty='l2', C=C, max_iter=1000).fit(X_train, y_train)
    f1_scores.append(f1_score(y_test, model.predict(X_test)))
    n_large_coefs.append(np.sum(np.abs(model.coef_[0]) > 0.1))
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(14,5))
ax1.plot(C_values,f1_scores,'o-'); ax1.set_xscale('log'); ax1.set_xlabel('C'); ax1.set_ylabel('F1 Score'); ax1.set_title('C vs F1 Score')
ax2.plot(C_values,n_large_coefs,'o-',color='coral'); ax2.set_xscale('log'); ax2.set_xlabel('C'); ax2.set_ylabel('Large Coefficients'); ax2.set_title('C vs Coefficient Magnitude'); plt.show()

## Analysis
Regularization constrains coefficient magnitude, reducing overfitting. L1 can also remove features by setting weights exactly to zero, while L2 typically retains all features with smaller weights. The preferred production model should balance test metrics, stability, and interpretability; the results table identifies that choice after execution.